# Turn-Taking Fusion Experiments

### Note that Early Fusion will only work for unimodal data -- or data collected at the same Hz!
#### Also please note you will have to configure a lot of the paths yourself
Thin driver for dyadic-fusion experiments on the turn-taking corpus. All heavy lifting (dataset classes, model classes, training helpers, τ-sweep) lives in `fusion_lib.py`. This notebook just:

1. Declares a modality registry (currently set up for CPC and OpenFace).
2. Declares a dict of experiments — each experiment is `{streams, fusion}` where `streams = [{modality, role}, ...]` and `fusion ∈ {'unimodal', 'early', 'neural_concat'}`.
3. Runs each experiment, collects results, and generates a τ-sweep curve.



## Section 0 — Setup

## init Google Drive


In [24]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [25]:
cd '535_project_data'

[Errno 2] No such file or directory: '535_project_data'
/content


## Init Local

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
import torch
import fusion_lib as fl

In [ ]:
# Dry-run / small-subset mode. Set to None to use all available samples.
# For smoke tests, try MAX_SAMPLES_PER_SPLIT = 200 to verify plumbing end-
# to-end in ~30 s before committing to a full run. Truncation is preceded
# by a seed-deterministic shuffle per split so the subset is representative.
MAX_SAMPLES_PER_SPLIT = None

# Model / training hyperparameters. Defaults match the CSCI-535 practicum
# (multimodal_fusion_practicum.ipynb): GRU_UNITS=64, EPOCHS=30, BATCH_SIZE=32,
# LR=1e-3, patience=4, dropout=0.3, num_layers=3 (EarlyFusion only).

HIDDEN_SIZE       = 64
DROPOUT           = 0.3
NUM_LAYERS_EARLY  = 3            # GRU stack depth for EarlyFusion only
EPOCHS            = 30           # max epochs; early stopping may trigger earlier
BATCH_SIZE        = 32
LEARNING_RATE     = 1e-3
PATIENCE          = 4            # early-stop after N epochs w/o val-F1 gain
NUM_WORKERS       = 0 
SEED              = 42

In [ ]:
RUN_ENV = "local"  # "colab" or "local"

if RUN_ENV == "colab":
    BASE = Path("/content/drive/MyDrive/535_project_data")
    CPC_DIR = BASE / "data/spliced/cpc"
    OPENFACE_DIR = BASE / "data/spliced/openface_new"
    LABELS_DIR = BASE / "data/spliced/labels_tau"
    MANIFEST_PATH = BASE / "data/spliced/manifests/manifest_cpc_available.csv"
    COORDINATION_CSV_PATH = BASE / "data/spliced/coordination/openface/openface/all_coordination_features_tau_0400.csv"
    COORDINATION_CONTINUOUS_DIR = BASE / "data/spliced/coordination/openface_continuous_arrays"

elif RUN_ENV == "local":
    BASE = Path("/Users/kaitlinzareno/Desktop/csci535/csci535-project")
    CPC_DIR = BASE / "data/spliced/cpc"
    OPENFACE_DIR = BASE / "data/spliced/openface_new"
    LABELS_DIR = BASE / "data/labels_tau"
    MANIFEST_PATH = BASE / "data/manifests/manifest_cpc_available.csv"
    COORDINATION_CSV_PATH = BASE / "data/coordination/openface/all_coordination_features_tau_0400.csv"
    COORDINATION_CONTINUOUS_DIR = BASE / "data/coordination/openface_continuous_arrays"

TRAIN_LABELS_PATH = LABELS_DIR / "labels_tau_0400.json"

OPENFACE_DIM = 23
COORDINATION_CONTINUOUS_DIM = OPENFACE_DIM  # wcc: shape (D = 23, 2 * max_lag = 10 + 1)

ABLATIONS = ['standard', 'ssa', 'coordination', 'csa'] #standard (no coordination, GRU), Standard w/ Self Attention, Coordination, Coordination + Self Attention
ABLATION = ABLATIONS[3] 

In [ ]:

print(f'manifest:         {MANIFEST_PATH}')
print(f'train labels:     {TRAIN_LABELS_PATH}')
print(f'dry-run cap:      {MAX_SAMPLES_PER_SPLIT}')

## Section 1 — Modality registry

Each entry: `{dir, feature_dim, frame_rate_hz}`. `frame_rate_hz` is metadata (not consumed by the loader) — kept for documentation. Add new modalities by appending entries and referencing them in experiment configs.

The on-disk layout each modality must satisfy:

```
<modality.dir>/
  <interaction_id>_<participant_id>/
    NNNN.NN-NNNN.NN_<interaction_id>_<participant_id>.npy
    ...
```

### Standard (no coordination)

In [14]:
MODALITY_REGISTRY = fl.make_modality_registry({
    'cpc': {
        'dir': CPC_DIR,
        'feature_dim': 256,
        'frame_rate_hz': 100.0,
    },
    'openface': {
        'dir': OPENFACE_DIR,
        'feature_dim': 23,
        'frame_rate_hz': 30.0,
    },
})
for name, cfg in MODALITY_REGISTRY.items():
    print(f'  {name}: dim={cfg["feature_dim"]}  rate={cfg["frame_rate_hz"]} Hz  dir={cfg["dir"]}')

  cpc: dim=256  rate=100.0 Hz  dir=/content/drive/MyDrive/535_project_data/data/spliced/cpc
  openface: dim=23  rate=30.0 Hz  dir=/content/drive/MyDrive/535_project_data/data/spliced/openface_new


### + coordination

In [ ]:
MODALITY_REGISTRY = fl.make_modality_registry_coordination({
    "cpc": {
        "kind": "spliced",
        "dir": str(CPC_DIR),
        "feature_dim": 256,
        "frame_rate_hz": 100.0,
    },

    "openface": {
        "kind": "spliced",
        "dir": str(OPENFACE_DIR),
        "feature_dim": 23,
        "frame_rate_hz": 30.0,
    },

    # Alias used by your coordination GRU configs.
    "coordination_openface": {
        "kind": "coordination",
        "coordination_mode": "summary",
        "feature_dim": len(fl.COORDINATION_FEATURE_COLUMNS),
        "frame_rate_hz": 0.0,
    },

    # Explicit summary name.
    "coordination_openface_summary": {
        "kind": "coordination",
        "coordination_mode": "summary",
        "feature_dim": len(fl.COORDINATION_FEATURE_COLUMNS),
        "frame_rate_hz": 0.0,
    },

    # Continuous WCC stream for CSA.
    "coordination_openface_continuous": {
        "kind": "coordination",
        "coordination_mode": "continuous",
        "dir": str(COORDINATION_CONTINUOUS_DIR),
        "feature_dim": COORDINATION_CONTINUOUS_DIM,
        "frame_rate_hz": 0.0,
    },
})

for name, cfg in MODALITY_REGISTRY.items():
    print(f'  {name}: dim={cfg["feature_dim"]}  rate={cfg["frame_rate_hz"]} Hz  dir={cfg["dir"]}')

  cpc: dim=256  rate=100.0 Hz  dir=/content/drive/MyDrive/535_project_data/data/spliced/cpc
  openface: dim=23  rate=30.0 Hz  dir=/content/drive/MyDrive/535_project_data/data/spliced/openface_new
  coordination_openface: dim=19  rate=0.0 Hz  dir=


## Section 2 — Load manifest + label sanity

In [33]:
manifest_rows = fl.load_manifest(str(MANIFEST_PATH))
print(f'manifest rows: {len(manifest_rows)}')
print(f'splits:        {dict((sp, sum(1 for r in manifest_rows if r["split"] == sp)) for sp in ("train", "val", "test"))}')

# Cross-check label coverage against the manifest.
train_labels = fl.load_labels(str(TRAIN_LABELS_PATH))
samples = fl.enumerate_samples(manifest_rows, train_labels)
summary = fl.summarize_samples(samples)
print(f'\ntotal training-usable samples at τ={TRAIN_TAU_MS} ms: {summary["total"]}')
print(f'per-split: {summary["per_split"]}')
for split, dist in summary['per_split_class'].items():
    print(f'  {split:5s}  class distribution: {dist}')

manifest rows: 222
splits:        {'train': 155, 'val': 33, 'test': 34}

total training-usable samples at τ=400 ms: 63598
per-split: {'val': 9729, 'train': 43684, 'test': 10185}
  val    class distribution: {'HOLD': 8667, 'BACKCHANNEL': 739, 'YIELD': 323}
  train  class distribution: {'HOLD': 39429, 'YIELD': 1323, 'BACKCHANNEL': 2932}
  test   class distribution: {'BACKCHANNEL': 719, 'HOLD': 9191, 'YIELD': 275}


## Section 3 — Experiment definitions

Each experiment declares its streams (modality + role) and fusion family.

**Fusion families supported:**
- `unimodal` — single stream, single GRU + FC.
- `early` — N streams concatenated on the feature axis, single 3-layer GRU.
- `neural_concat` — exactly 2 streams, two parallel GRUs, concat final hidden states, FC head.

**Roles supported per stream:**
- `speaker` — the floor holder (perspective participant).
- `listener` — the co-participant (resolved via manifest).

Add / remove experiments by editing `EXPERIMENTS` below.

DEFINITIONS:
- standard:     fl.run_experiment              + no attention args
- ssa:          fl.run_experiment              + attention args
- coordination: fl.run_experiment_coordination + no attention args
- csa:          fl.run_experiment_coordination + attention args

### Standard (no coordination, GRU)

In [ ]:
EXPERIMENTS = {
    'cpc_both_early': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
        ],
        'fusion':  'early',
    },
    'cpc_both_neural_concat': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
        ],
        'fusion':  'neural_concat',
    },
    'cpc_speaker_of_listener': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion':  'neural_concat',
    },
    'cpc_speaker_of_dyad': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
        ],
        'fusion': 'neural_concat',
    },
    'full_dyad': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'cpc',      'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion': 'neural_concat',
    }
}

print(f'configured experiments: {len(EXPERIMENTS)}')
for name, cfg in EXPERIMENTS.items():
    print(f'  {name:30s}  fusion={cfg["fusion"]:14s}  streams={cfg["streams"]}')

### Standard + Self Attention (SSA)

In [ ]:
ATTENTION_EXPERIMENTS = {
    'cpc_both_attention': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
        ],
        'fusion':  'self_attention',
    },
    'of_speaker_of_listener_attention': {
        'streams': [
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion':  'self_attention',
    },
    'cpc_speaker_of_listener_attention': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion':  'self_attention',
    },
    'cpc_speaker_of_dyad_attention': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
        ],
        'fusion': 'self_attention',
    },
    'full_dyad_attention': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'cpc',      'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion': 'self_attention',
    }
    }


### Experiments w/ Coordination

In [ ]:
EXPERIMENTS_COORDINATION ={
     'cpc_speaker_plus_coordination': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'coordination_openface', 'role': 'speaker'},
        ],
        'fusion': 'neural_concat',
    },
    
    #ONLY TEST OPENFSACE AND COORD LOCAL
        "openface_with_coord_neural_concat": {
        "streams": [
            {"modality": "openface", "role": "speaker"},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        "fusion": "neural_concat",
    },

    #ONLY TEST OPENFSACE AND COORD LOCAL
      "openface_BOTH_with_coord_neural_concat": {
        "streams": [
            {"modality": "openface", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        "fusion": "neural_concat",
    },

    'cpc_speaker_of_listener_coord': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        'fusion':  'neural_concat',
    },
     
     'cpc_speaker_both_faces_coord': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        'fusion':  'neural_concat',
    },

    'full_dyad_plus_coordination': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {'modality': 'coordination_openface', 'role': 'speaker'},
        ],
        'fusion': 'neural_concat',
    },
}

print(f'configured experiments: {len(EXPERIMENTS_COORDINATION)}')
for name, cfg in EXPERIMENTS_COORDINATION.items():
    print(f'  {name:30s}  fusion={cfg["fusion"]:14s}  streams={cfg["streams"]}')

configured experiments: 3
  cpc_speaker_plus_coordination   fusion=neural_concat   streams=[{'modality': 'cpc', 'role': 'speaker'}, {'modality': 'coordination_openface', 'role': 'speaker'}]
  openface_with_coord_neural_concat  fusion=neural_concat   streams=[{'modality': 'openface', 'role': 'speaker'}, {'modality': 'coordination_openface', 'role': 'speaker'}]
  full_dyad_plus_coordination     fusion=neural_concat   streams=[{'modality': 'cpc', 'role': 'speaker'}, {'modality': 'cpc', 'role': 'listener'}, {'modality': 'openface', 'role': 'speaker'}, {'modality': 'openface', 'role': 'listener'}, {'modality': 'coordination_openface', 'role': 'speaker'}]


### Coodination + Self Attention (CSA)

In [ ]:
ATTENTION_EXPERIMENTS_COORDINATION = {
    "cpc_speaker_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 2],
    },

    "cpc_speaker_cpc_listener_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "cpc", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "cpc_speaker_of_speaker_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "openface", "role": "speaker"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "openface_coord_self_attention": {
        "streams": [
            {"modality": "openface", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "cpc_speaker_of_listener_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "full_dyad_plus_coordination_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "cpc", "role": "listener"},
            {"modality": "openface", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 4, 4, 2],
    },
}

print(f'configured experiments: {len(ATTENTION_EXPERIMENTS_COORDINATION)}')
for name, cfg in ATTENTION_EXPERIMENTS_COORDINATION.items():
    print(f'  {name:30s}  fusion={cfg["fusion"]:14s}  streams={cfg["streams"]}')

## Section 4 — Run experiments

Trains one model per experiment at the canonical τ (default 400 ms), keeps the best-val-macro-F1 checkpoint, and evaluates on test. Early stopping kicks in after `PATIENCE` epochs without a val-F1 improvement (caps training well under `EPOCHS`). Results are accumulated in `results` and reused in Section 5 for the τ-sweep (input features are τ-invariant; only labels change).

#### General Runner

In [ ]:
import json
import time
import torch
from pathlib import Path
from tqdm.auto import tqdm

def make_json_safe_result(res):
    safe = {}
    for k, v in res.items():
        if k == "model_state":
            continue
        safe[k] = v
    return safe


def run_ablation_block(
    *,
    ablation_name,
    experiments,
    use_coordination,
    use_attention,
    out_root,
    modality_registry,
    manifest_rows,
    train_labels_path,
    coordination_csv_path=None,
    hidden_size=64,
    dropout=0.3,
    num_layers_early=3,
    epochs=30,
    batch_size=32,
    learning_rate=1e-3,
    patience=4,
    num_workers=0,
    seed=42,
    max_samples_per_split=None,
):
    run_id = time.strftime("%Y%m%d_%H%M%S")
    out_dir = Path(out_root) / ablation_name / run_id
    out_dir.mkdir(parents=True, exist_ok=True)

    results = {}

    for name, cfg in tqdm(experiments.items(), desc=ablation_name):
        print(f"\nRunning {name}")

        try:
            runner = fl.run_experiment_coordination if use_coordination else fl.run_experiment

            kwargs = dict(
                name=name,
                streams=cfg["streams"],
                fusion=cfg["fusion"],
                modality_registry=modality_registry,
                manifest_rows=manifest_rows,
                labels_path=str(train_labels_path),
                hidden_size=hidden_size,
                dropout=dropout,
                num_layers_early=num_layers_early,
                epochs=epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience,
                num_workers=num_workers,
                seed=seed,
                max_samples_per_split=max_samples_per_split,
            )

            if use_coordination:
                kwargs.update(
                    coordination_csv_path=str(coordination_csv_path) if coordination_csv_path else None,
                    missing_coordination="zeros",
                )

            if use_attention:
                kwargs.update(
                    attention_dim=cfg.get("attention_dim", hidden_size),
                    attention_heads=cfg.get("attention_heads_by_modality", cfg.get("attention_heads", 4)),
                    attention_layers=cfg.get("attention_layers", 2),
                    attention_pooling=cfg.get("attention_pooling", "mean"),
                )

            res = runner(**kwargs)
            results[name] = res

            json_path = out_dir / f"{name}_results.json"
            model_path = out_dir / f"{name}_model_state.pt"

            with open(json_path, "w") as f:
                json.dump(make_json_safe_result(res), f, indent=2, default=str)

            torch.save(res["model_state"], model_path)

            print(f"Saved {name}:")
            print(f"  {json_path}")
            print(f"  {model_path}")

        except Exception as e:
            print(f"FAILED: {name}")
            print(repr(e))

            partial_path = out_dir / "partial_results.json"
            with open(partial_path, "w") as f:
                json.dump(
                    {k: make_json_safe_result(v) for k, v in results.items()},
                    f,
                    indent=2,
                    default=str,
                )
            raise

    print("\n=== summary at training τ ===")
    for name, res in results.items():
        print(
            f'{name:40s} test macro-F1 = {res["test_eval"]["macro_f1"]:.4f} '
            f'per-class = {res["test_eval"]["per_class_f1"]}'
        )

    print("Run saved to:", out_dir)
    return results, out_dir

In [ ]:
OUT_ROOT = BASE / "fusion_runs" / "summary"

#### run standard

In [ ]:
standard_results, standard_out_dir = run_ablation_block(
    ablation_name="standard",
    experiments=EXPERIMENTS,
    use_coordination=False,
    use_attention=False,
    out_root=OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    train_labels_path=TRAIN_LABELS_PATH,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

#### run ssa

In [ ]:
ssa_results, ssa_out_dir = run_ablation_block(
    ablation_name="ssa",
    experiments=ATTENTION_EXPERIMENTS,
    use_coordination=False,
    use_attention=True,
    out_root=OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    train_labels_path=TRAIN_LABELS_PATH,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

#### run coordination

In [ ]:
coordination_results, coordination_out_dir = run_ablation_block(
    ablation_name="coordination",
    experiments=EXPERIMENTS_COORDINATION,
    use_coordination=True,
    use_attention=False,
    out_root=OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    train_labels_path=TRAIN_LABELS_PATH,
    coordination_csv_path=COORDINATION_CSV_PATH,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

#### run csa

In [ ]:
csa_results, csa_out_dir = run_ablation_block(
    ablation_name="csa",
    experiments=ATTENTION_EXPERIMENTS_COORDINATION,
    use_coordination=True,
    use_attention=True,
    out_root=OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    train_labels_path=TRAIN_LABELS_PATH,
    coordination_csv_path=COORDINATION_CSV_PATH,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

## Section 5 — τ-sweep

Evaluates each trained model against every per-τ label file. Features are τ-invariant, so the model weights don't change — only the label-to-basename mapping swaps. The τ-sweep uses the same class-weighted CrossEntropyLoss that was used during training (weights derived from the train-split at training-τ), so per-τ loss numbers are comparable to the training history.

#### Tau Runner

In [ ]:
def run_tau_sweep_block(
    *,
    ablation_name,
    results,
    use_coordination,
    out_root,
    modality_registry,
    manifest_rows,
    labels_dir,
    coordination_csv_path=None,
    batch_size=32,
    num_workers=0,
    seed=42,
    max_samples_per_split=None,
):
    run_id = time.strftime("%Y%m%d_%H%M%S")
    out_dir = Path(out_root) / ablation_name / run_id
    out_dir.mkdir(parents=True, exist_ok=True)

    tau_curves = {}

    for name, res in results.items():
        print(f"\n=== τ-sweep: {name} ===")

        try:
            if use_coordination:
                tau_res = fl.sweep_tau_coordination_safe(
                    experiment_result=res,
                    modality_registry=modality_registry,
                    manifest_rows=manifest_rows,
                    labels_dir=str(labels_dir),
                    tau_grid_ms=fl.DEFAULT_TAU_GRID_MS,
                    batch_size=batch_size,
                    num_workers=num_workers,
                    seed=seed,
                    max_samples_per_split=max_samples_per_split,
                    coordination_csv_path=str(coordination_csv_path) if coordination_csv_path else None,
                )
            else:
                tau_res = fl.sweep_tau(
                    experiment_result=res,
                    modality_registry=modality_registry,
                    manifest_rows=manifest_rows,
                    labels_dir=str(labels_dir),
                    tau_grid_ms=fl.DEFAULT_TAU_GRID_MS,
                    batch_size=batch_size,
                    num_workers=num_workers,
                    seed=seed,
                    max_samples_per_split=max_samples_per_split,
                )

            tau_curves[name] = tau_res

            tau_path = out_dir / f"{name}_tau_curves.json"
            with open(tau_path, "w") as f:
                json.dump(tau_res, f, indent=2)

            print(f"Saved τ results → {tau_path}")

        except Exception as e:
            print(f"FAILED τ-sweep: {name}")
            print(repr(e))

            partial_path = out_dir / "partial_tau_curves.json"
            with open(partial_path, "w") as f:
                json.dump(tau_curves, f, indent=2)

            raise

    print("τ-sweeps saved to:", out_dir)
    return tau_curves, out_dir

In [ ]:
TAU_OUT_ROOT = BASE / "fusion_runs" / "tau_sweeps"

#### Tau Standard

In [ ]:
standard_tau_curves, _ = run_tau_sweep_block(
    ablation_name="standard",
    results=standard_results,
    use_coordination=False,
    out_root=TAU_OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    labels_dir=LABELS_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

#### Tau SSA

In [ ]:
ssa_tau_curves, _ = run_tau_sweep_block(
    ablation_name="ssa",
    results=ssa_results,
    use_coordination=False,
    out_root=TAU_OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    labels_dir=LABELS_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

#### Tau Coordination

In [ ]:
coord_tau_curves, _ = run_tau_sweep_block(
    ablation_name="coordination",
    results=coordination_results,
    use_coordination=True,
    out_root=TAU_OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    labels_dir=LABELS_DIR,
    coordination_csv_path=COORDINATION_CSV_PATH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

#### TAU CSA

In [ ]:
csa_tau_curves, _ = run_tau_sweep_block(
    ablation_name="csa",
    results=csa_results,
    use_coordination=True,
    out_root=TAU_OUT_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    labels_dir=LABELS_DIR,
    coordination_csv_path=COORDINATION_CSV_PATH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)

## Section 6 — τ-curve plot

In [ ]:
import matplotlib.pyplot as plt

tau_curves_list = [standard_tau_curves,ssa_tau_curves,coord_tau_curves, csa_tau_curves]

#plot tau depending on ablation
if ABLATION == 'standard':
    tau_curves = tau_curves_list[0] 
elif ABLATION == 'ssa':
    tau_curves = tau_curves_list[1]
elif ABLATION == 'coordination':
    tau_curves = tau_curves_list[2]
elif ABLATION == 'csa':
    tau_curves = tau_curves_list[3]

fig, ax = plt.subplots(1, 1, figsize=(8, 5))

for name, curve in tau_curves.items():
    taus = sorted(curve.keys())
    f1s = [curve[t]['macro_f1'] for t in taus]
    ax.plot(taus, f1s, marker='o', label=name)

ax.set_xlabel('τ (ms) — prediction horizon')
ax.set_ylabel('test macro-F1')
ax.set_title(f'Turn-taking horizon curve  (trained @ τ={TRAIN_TAU_MS} ms)')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

## Section 7 — Save Results Data
### (it is excessive, but we won't regret having too much to make figs from I think)

Write a JSON of results (minus `model_state`, which is a tensor dict) for later comparison across notebook runs.

Section 7 — Persist detailed results for offline analysis & figure-making

Why so much detail: the in-memory `results` and `tau_curves` dicts only carry
scalar summaries (macro-F1, per-class F1). To make confusion-matrix heatmaps,
#ROC/AUC curves, calibration plots, per-interaction breakdowns, or any kind of
error analysis later, we need the per-sample softmax probabilities, the
predicted/true class per sample, sample identifiers, and full per-class
precision/recall/F1/support — at every τ we evaluate.

Output layout:
   model_input/fusion_runs/<run_id>/run_index.json    — overall run metadata
   model_input/fusion_runs/<run_id>/<exp_name>.json   — full per-experiment results

In [ ]:
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1" # get rid of debugger output

In [ ]:
import json, os, sys, time
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import torch

from pathlib import Path

RUN_ID = time.strftime('%Y%m%d_%H%M%S')
#OUT_DIR = PROJ / 'model_input' / 'fusion_runs' / RUN_ID

OUT_DIR = Path(f'{RESULTS_DIR}/{MANIFEST_TYPE}/s7/{ABLATION}/{RUN_ID}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = fl.resolve_device('auto')

LABELS_DIR = Path(LABELS_DIR)

OPENFACE_DIR = Path(OPENFACE_DIR)

MODALITY_REGISTRY["openface"]["dir"] = str(OPENFACE_DIR)

print(MODALITY_REGISTRY["openface"])

if ABLATION == 'standard':
    s7_experiments = EXPERIMENTS
    s7_results = standard_results
elif ABLATION == 'ssa':
    s7_experiments =  ATTENTION_EXPERIMENTS
    s7_results = ssa_results
elif ABLATION == 'coordination':
    s7_experiments = EXPERIMENTS_COORDINATION
    s7_results = coordination_results
elif ABLATION == 'csa':
    s7_experiments = ATTENTION_EXPERIMENTS_COORDINATION
    s7_results = csa_results

In [ ]:
# ---------------------------------------------------------------------------
# Helpers — kept local to Section 7 so we don't have to touch fusion_lib.
# ---------------------------------------------------------------------------

def _inference_with_probs(model, loader, device):
    """Run inference, capture per-sample softmax probs + argmax preds + labels."""
    model.eval()
    probs_chunks, preds, labels = [], [], []
    with torch.inference_mode():
        for streams, ys in loader:
            streams = [t.to(device) for t in streams]
            logits = model(streams)
            p = torch.softmax(logits, dim=-1).cpu().numpy()
            probs_chunks.append(p)
            preds.extend(p.argmax(-1).tolist())
            labels.extend(ys.cpu().tolist())
    probs = np.concatenate(probs_chunks, axis=0) if probs_chunks else np.zeros((0, fl.NUM_CLASSES), dtype=np.float32)
    return probs, preds, labels


def _confusion_matrix(preds, labels, num_classes=fl.NUM_CLASSES):
    """Rows = true class, cols = predicted class. Counts."""
    cm = [[0] * num_classes for _ in range(num_classes)]
    for p, y in zip(preds, labels):
        if 0 <= y < num_classes and 0 <= p < num_classes:
            cm[y][p] += 1
    return cm


def _per_class_metrics(preds, labels, num_classes=fl.NUM_CLASSES):
    """precision, recall, F1, support, plus raw tp/fp/fn for downstream stats."""
    out = {}
    for c in range(num_classes):
        tp = sum(1 for p, y in zip(preds, labels) if p == c and y == c)
        fp = sum(1 for p, y in zip(preds, labels) if p == c and y != c)
        fn = sum(1 for p, y in zip(preds, labels) if p != c and y == c)
        support = sum(1 for y in labels if y == c)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        out[fl.CLASS_NAMES[c]] = {
            'precision': prec, 'recall': rec, 'f1': f1,
            'support': support, 'tp': tp, 'fp': fp, 'fn': fn,
        }
    return out


def _aggregate_macro(per_class: dict) -> dict:
    """Macro precision/recall/F1 (unweighted mean over classes) + weighted F1
    (mean over classes weighted by support — useful when you want a single
    number that respects class imbalance)."""
    classes = list(per_class.values())
    n = len(classes) or 1
    macro_p  = sum(c['precision'] for c in classes) / n
    macro_r  = sum(c['recall']    for c in classes) / n
    macro_f1 = sum(c['f1']        for c in classes) / n
    total_support = sum(c['support'] for c in classes) or 1
    weighted_f1 = sum(c['f1'] * c['support'] for c in classes) / total_support
    return {
        'macro_precision': macro_p,
        'macro_recall':    macro_r,
        'macro_f1':        macro_f1,
        'weighted_f1':     weighted_f1,
    }


def _grouped_metrics(samples, preds, labels, key_fn):
    """Slice (samples, preds, labels) by `key_fn(sample)` and compute per-group
    n / macro-F1 / per-class metrics / confusion matrix. Used for per-interaction,
    per-participant, per-relationship breakdowns."""
    groups = defaultdict(lambda: {'preds': [], 'labels': []})
    for s, p, y in zip(samples, preds, labels):
        k = key_fn(s)
        groups[k]['preds'].append(p)
        groups[k]['labels'].append(y)
    out = {}
    for k, d in groups.items():
        per_class = _per_class_metrics(d['preds'], d['labels'])
        agg = _aggregate_macro(per_class)
        out[k] = {
            'n_samples': len(d['preds']),
            **agg,
            'per_class': per_class,
            'confusion_matrix': _confusion_matrix(d['preds'], d['labels']),
        }
    return out


# Manifest-derived lookup for relationship / participant slicing.
_iid_to_relationship = {r['interaction_id']: r.get('relationship', 'unknown')
                        for r in manifest_rows}
_iid_to_relationship_detail = {r['interaction_id']: r.get('relationship_detail', 'unknown')
                                for r in manifest_rows}


In [ ]:

# ---------------------------------------------------------------------------
# Per-experiment full re-evaluation across τ-grid WITH PARTIAL SAVES
# ---------------------------------------------------------------------------

detailed_results = {}

print("results keys:", list(s7_results.keys()))

expected = set(s7_experiments.keys())
actual = set(s7_results.keys())

print("expected:", expected)
print("actual:", actual)
print("missing:", expected - actual)

# Optional: use smaller grid for debugging
TAU_GRID = fl.DEFAULT_TAU_GRID_MS
# TAU_GRID = (200, 400, 800)

for exp_name in s7_experiments.keys():

    if exp_name not in s7_results:
        print(f"SKIPPING {exp_name}: not found in coordination_results")
        continue

    exp_res = s7_results[exp_name]

    if "model_state" not in exp_res:
        print(f"SKIPPING {exp_name}: no model_state")
        continue

    print(f"\nSaving detailed results for: {exp_name}")

    cfg = exp_res["config"]
    stream_cfg = cfg["streams"]
    fusion = cfg["fusion"]
    stream_dims = exp_res["stream_dims"]

    model = fl.build_model(
        fusion,
        stream_dims,
        hidden_size=cfg["hidden_size"],
        dropout=cfg.get("dropout", 0.3),
        num_layers_early=cfg.get("num_layers_early", 3),
    ).to(DEVICE)

    model.load_state_dict(exp_res["model_state"])

    train_labels_for_weights = fl.load_labels(cfg["labels_path"])
    train_samples_for_weights = [
        s for s in fl.enumerate_samples(manifest_rows, train_labels_for_weights)
        if s["split"] == "train"
    ]

    class_weights_used = (
        fl.compute_class_weights(train_samples_for_weights)
        .cpu()
        .tolist()
    )

    criterion = torch.nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights_used, dtype=torch.float32, device=DEVICE)
    )

    history = exp_res["train_history"]
    best_epoch_entry = max(history, key=lambda h: h["val_macro_f1"]) if history else None
    n_params = sum(int(p.numel()) for p in model.parameters())

    coordination_lookup = fl.load_coordination_features(
        str(COORDINATION_CSV_PATH),
        feature_cols=fl.COORDINATION_FEATURE_COLUMNS,
    )

    tau_results = {}

    detailed_results[exp_name] = {
        "experiment_name": exp_name,
        "config": cfg,
        "stream_dims": stream_dims,
        "device": exp_res["device"],
        "n_params": n_params,
        "class_weights_used": class_weights_used,
        "samples_summary": exp_res["samples_summary"],
        "train_history": history,
        "best_epoch": best_epoch_entry["epoch"] if best_epoch_entry else None,
        "best_val_macro_f1": best_epoch_entry["val_macro_f1"] if best_epoch_entry else None,
        "epochs_completed": len(history),
        "tau_results": tau_results,
    }

    out_path = OUT_DIR / f"{exp_name}.json"
    partial_path = OUT_DIR / f"{exp_name}_partial.json"

    for tau_ms in TAU_GRID:
        try:
            print(f"\nτ={tau_ms}")

            labels_path = LABELS_DIR / f"labels_tau_{tau_ms:04d}.json"
            if not labels_path.exists():
                print(f"  labels missing, skipping: {labels_path}")
                continue

            labels_dict = fl.load_labels(str(labels_path))
            samples = fl.enumerate_samples(manifest_rows, labels_dict)
            test_samples = [s for s in samples if s["split"] == "test"]

            if not test_samples:
                print(f"  no test samples for τ={tau_ms}, skipping")
                continue

            loader = fl.make_dataloader_coordination(
                test_samples,
                stream_cfg,
                MODALITY_REGISTRY,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                coordination_lookup=coordination_lookup,
                coordination_feature_dim=len(fl.COORDINATION_FEATURE_COLUMNS),
                missing_coordination="zeros",
            )

            probs, preds, labels = _inference_with_probs(model, loader, DEVICE)

            per_class = _per_class_metrics(preds, labels)
            agg = _aggregate_macro(per_class)

            with torch.inference_mode():
                test_loss = 0.0
                n = 0

                for streams_batch, ys in loader:
                    streams_batch = [t.to(DEVICE) for t in streams_batch]
                    ys = ys.to(DEVICE)

                    logits = model(streams_batch)
                    test_loss += criterion(logits, ys).item() * ys.size(0)
                    n += ys.size(0)

                test_loss = test_loss / max(n, 1)

            sample_records = [
                {
                    "basename": s["basename"],
                    "interaction_id": s["interaction_id"],
                    "speaker_file_id": s["speaker_file_id"],
                    "listener_file_id": s["listener_file_id"],
                    "window_start_s": s["start_s"],
                    "window_end_s": s["end_s"],
                    "split": s["split"],
                    "label": int(y),
                    "pred": int(p),
                    "probs": [float(x) for x in row],
                    "correct": int(p) == int(y),
                }
                for s, p, y, row in zip(test_samples, preds, labels, probs)
            ]

            tau_results[str(tau_ms)] = {
                "n_test_samples": len(test_samples),
                "test_loss": test_loss,
                **agg,
                "per_class": per_class,
                "confusion_matrix": _confusion_matrix(preds, labels),
                "class_distribution_test": dict(Counter(labels)),
                "per_interaction": _grouped_metrics(
                    test_samples,
                    preds,
                    labels,
                    key_fn=lambda s: s["interaction_id"],
                ),
                "per_speaker": _grouped_metrics(
                    test_samples,
                    preds,
                    labels,
                    key_fn=lambda s: s["speaker_file_id"],
                ),
                "per_relationship": _grouped_metrics(
                    test_samples,
                    preds,
                    labels,
                    key_fn=lambda s: _iid_to_relationship.get(
                        s["interaction_id"], "unknown"
                    ),
                ),
                "per_relationship_detail": _grouped_metrics(
                    test_samples,
                    preds,
                    labels,
                    key_fn=lambda s: _iid_to_relationship_detail.get(
                        s["interaction_id"], "unknown"
                    ),
                ),
                "sample_records": sample_records,
            }

            detailed_results[exp_name]["tau_results"] = tau_results

            with open(out_path, "w") as f:
                json.dump(detailed_results[exp_name], f, indent=2, default=str)

            print(f"  [saved τ={tau_ms}] → {out_path}")

        except Exception as e:
            print(f"  FAILED at τ={tau_ms}")
            print(repr(e))

            detailed_results[exp_name]["tau_results"] = tau_results

            with open(partial_path, "w") as f:
                json.dump(detailed_results[exp_name], f, indent=2, default=str)

            print(f"  [partial save] → {partial_path}")
            raise

    with open(out_path, "w") as f:
        json.dump(detailed_results[exp_name], f, indent=2, default=str)

    print(f"  wrote final {out_path}")
    print(f"  size: {out_path.stat().st_size / 1024:.1f} KB")


# ---------------------------------------------------------------------------
# Run-level index
# ---------------------------------------------------------------------------

run_index = {
    "run_id": RUN_ID,
    "timestamp": datetime.now().isoformat(),
    "manifest_path": str(MANIFEST_PATH),
    "train_labels_path": str(TRAIN_LABELS_PATH),
    "train_tau_ms": TRAIN_TAU_MS,
    "tau_grid_ms": list(TAU_GRID),
    "modality_registry": MODALITY_REGISTRY,
    "global_hyperparams": {
        "hidden_size": HIDDEN_SIZE,
        "dropout": DROPOUT,
        "num_layers_early": NUM_LAYERS_EARLY,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "seed": SEED,
        "max_samples_per_split": MAX_SAMPLES_PER_SPLIT,
    },
    "experiments": list(detailed_results.keys()),
    "experiment_summary": {
        name: {
            "fusion": d["config"]["fusion"],
            "streams": d["config"]["streams"],
            "n_params": d["n_params"],
            "best_epoch": d["best_epoch"],
            "epochs_run": d["epochs_completed"],
            "test_macro_f1_at_train_tau": d["tau_results"]
                .get(str(TRAIN_TAU_MS), {})
                .get("macro_f1"),
        }
        for name, d in detailed_results.items()
    },
    "environment": {
        "torch_version": torch.__version__,
        "python_version": sys.version.split()[0],
        "device": str(DEVICE),
        "platform": sys.platform,
    },
}

with open(OUT_DIR / "run_index.json", "w") as f:
    json.dump(run_index, f, indent=2, default=str)

print(f'  wrote {OUT_DIR / "run_index.json"}')

print(f"\nRun directory: {OUT_DIR}")
print("Files:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

{'kind': 'spliced', 'dir': '/content/drive/MyDrive/535_project_data/data/spliced/openface_new', 'feature_dim': 23, 'frame_rate_hz': 30.0}
coordination_results keys: ['cpc_speaker_plus_coordination', 'openface_with_coord_neural_concat', 'full_dyad_plus_coordination']
expected: {'openface_with_coord_neural_concat', 'cpc_speaker_plus_coordination', 'full_dyad_plus_coordination'}
actual: {'openface_with_coord_neural_concat', 'cpc_speaker_plus_coordination', 'full_dyad_plus_coordination'}
missing: set()

Saving detailed results for: cpc_speaker_plus_coordination

τ=100


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  FAILED at τ=100
FileNotFoundError('Caught FileNotFoundError in DataLoader worker process 1.\nOriginal Traceback (most recent call last):\n  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop\n    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]\n           ^^^^^^^^^^^^^^^^^^^^\n  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch\n    data = [self.dataset[idx] for idx in possibly_batched_index]\n            ~~~~~~~~~~~~^^^^^\n  File "/content/drive/MyDrive/535_project_data/scripts/fusion_lib.py", line 605, in __getitem__\n    arr = _load_feature_file(path_no_ext)\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/content/drive/MyDrive/535_project_data/scripts/fusion_lib.py", line 155, in _load_feature_file\n    raise FileNotFoundError(f"Missing feature file: {npy_path} or {json_path}")\nFileNotFoundError: Missing feature file: /content/drive/MyDrive/535_project_da

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/drive/MyDrive/535_project_data/scripts/fusion_lib.py", line 605, in __getitem__
    arr = _load_feature_file(path_no_ext)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/535_project_data/scripts/fusion_lib.py", line 155, in _load_feature_file
    raise FileNotFoundError(f"Missing feature file: {npy_path} or {json_path}")
FileNotFoundError: Missing feature file: /content/drive/MyDrive/535_project_data/data/spliced/cpc/V00_S1295_I